In [0]:
from datetime import datetime
import logging
import json
import ast
from dateutil.relativedelta import relativedelta

In [0]:
%run ../logging/logging_utils

In [0]:
RunId = dbutils.widgets.get("RunId")
snapshot_date = dbutils.widgets.get("snapshot_date")
cod_flowName = dbutils.widgets.get("cod_flowName")
env = dbutils.widgets.get("env")
cod_productName = dbutils.widgets.get("cod_productName")

In [0]:
err_widget = dbutils.widgets.get("Error") or "UnknownError"
state_widget = dbutils.widgets.get("State") or "UnknownState"
Error = f"{err_widget} - {state_widget}"

general_utils = LoggingUtils(spark)
logger = general_utils.set_logging()

config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
snapshot_date = general_utils.date_to_start_month(snapshot_date)
catalog_name = config['environment'][env]['catalog']
schema_name = config['environment'][env]['schema']

logger.info(f"Snapshot_date materialized: {snapshot_date}")

try:
    # logger.info(f"Log generation process INITIALIZING")
    # params = general_utils.get_pipeline_parameters()

    logger.info(f"Obtaining log processor reference")
    log_catalog, log_database, log_table = general_utils.get_log_processor_reference(
        catalog_name, "monitoring", config['logging']['processor']['table']
    )
except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

try:
    logger.info(f"Closing log with Error state on {log_table} table")
    
    general_utils.close_process_log(
        catalog=catalog_name, 
        database=log_database, 
        table=log_table, 
        metrica=cod_flowName, 
        cod_runID=RunId, 
        is_error=True, 
        error_msg=Error
    )
    
    logger.info(f"Log generation process END")
except Exception as e:
    logger.error(f"Log generation process ERROR as {e}")
    raise

In [0]:
# RunId = dbutils.widgets.get("RunId")
# snapshot_date = dbutils.widgets.get("snapshot_date")
# cod_flowName = dbutils.widgets.get("cod_flowName")
# env = dbutils.widgets.get("env")
# Error = dbutils.widgets.get("Error")+" - "+dbutils.widgets.get("State")

# general_utils = LoggingUtils(spark)
# logger = general_utils.set_logging()

# config = general_utils.load_config(general_utils.get_variable()['config_setup_path'])


# snapshot_date = general_utils.date_to_start_month(snapshot_date)
# catalog_name = config['environment'][env]['catalog']
# schema_name = config['environment'][env]['schema']
# logger.info(f"Snapshot_date materialized: {snapshot_date}")

# try:
#     logger.info(f"Log generation process INITIALIZING")

#     logger.info(f"Get parameters")
#     params = general_utils.get_pipeline_parameters()

#     logger.info(f"Obtaining log processor reference")
#     log_catalog, log_database, log_table = general_utils.get_log_processor_reference(catalog_name, 
#                                 "monitoring", 
#                                 config['logging']['processor']['table'])
# except Exception as e:
#     logger.error(f"Log generation process ERROR as {e}")
#     raise

# try:
#   logger.info(f"Log dateEnd population on {log_table} table")
#   general_utils.populate_monitor_log_date(catalog_name, log_database, log_table, cod_flowName, RunId, is_started=False)

#   logger.info(f"Log error population on {log_table} table")
#   general_utils.populate_log_error(catalog_name, log_database, log_table, cod_flowName, snapshot_date, Error, RunId, is_error=True)

#   logger.info(f"Log state population on {log_table} table")
#   general_utils.populate_log_state(catalog_name, log_database, log_table, cod_flowName, snapshot_date, RunId, is_error=True)
#   logger.info(f"Log generation process END")
# except Exception as e:
#     logger.error(f"Log generation process ERROR as {e}")
#     raise

In [0]:
dbutils.notebook.exit("Success!!")